<a href="https://colab.research.google.com/github/nonotoy/poysuwop/blob/main/01_Ain_Tokenizer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Poysuwop_Nr1 / Ainu Tokenizer
Transformer based tokenizer

### 0. Library install

In [ ]:
!pip install torch accelerate transformers tokenizers huggingface sentencepiece

### 1. Preprocess

In [81]:
# Library
import os
import glob
import json
import re
import collections
import unicodedata
import pprint

# Change Current Directory
os.chdir('/content/drive/MyDrive/Colab Notebooks/Poysuwop')

#### Load dataset

In [82]:
def alphabetCheck(text):
    for char in text:
        if not (char.isascii() and (char.isalpha() or char.isspace() or '=' or "'")):
            return False
    return True

In [53]:
# Fetch sentence using the 'ain' tag from .json file
# ['ain']       romanized Ainu sentence
# ['ain-kana']  Ainu sentence transcribed in Ainu Kana
# ['jpn']       Japanese translation

texts = []

verbalslip_pattern = r'(^| [a-zA-Z]*)(\.|x|X){2,}'

for f in glob.glob('./corpus_json/*.json'):

    # load .json
    data = json.load(open(str(f)))

    for key, value in data.items():

        if key != 'code' and key != 'title':
            sentence = value['ain']

            if len(sentence) > 0 and alphabetCheck(sentence) == True:

                # Verbal slip: Words with slips should be removed, since it may produce non-existing words
                # This procedure should be omitted from input to Transformer model in order not to create a non-sentence
                if re.search(verbalslip_pattern,sentence):
                    sentence = re.sub(verbalslip_pattern, ' ', sentence)
                    sentence = ' '.join(sentence.split())

                    texts.append(sentence)

In [54]:
##### Check charcters used in the dataset

charlist = [char for text in texts for char in list(text)]
charlist = [k for k, v in collections.Counter(charlist).items() if v > 1]

#charlist = [re.sub(r"[^a-zA-Z0-9 =]","",char) for char in charlist]
charlist_TBR = ''.join(str(x) for x in charlist)
char_TBR = re.sub(r"[a-zÞA-Z0-9 =]","",charlist_TBR)
print("Charcaters to be removed: {0}".format(char_TBR))

Charcaters to be removed: .'_(),?[]`"


#### Cleansing

In [55]:
# Affix cleansing
# Affix marker = is replaced as Þ

dicPrefixes = {
    'ku=':'kuþ ', #1sg.nom
    'k=':'kþ ', #1sg.nom
    'e=':'eþ ', #2sg.nom
    'eci=':'eciþ ', #2pl.nom
    'es=':'esþ ', #2pl.nom [Ishikar]
    'a=':'aþ ', #indefinite / inclusive 1pl.nom
    'an=':'anþ ', #indefinite / inclusive 1pl.nom
    'ci=':'ciþ ', #exclusive 1pl.nom
    'c=':'cþ ', #exclusive 1pl.nom
    'i=':'iþ ', #indefinite acc
    'en=':'enþ ', #1sg.acc
    'un=':'unþ ', #1pl.acc
}

dicSuffixes = {
    '=as':' þas',
    '=an':' þan',
}

def ainAffixCleanse(sentence, dic_chars, mode):
    # mode: 0 - prefix, 1 - suffix

    if mode == 0: #prefix
        for key in dic_chars.keys():
            sentence = re.sub(r'(\s|^)' + re.escape(key), r'\1' +dic_chars[key], sentence)

        # Special treatment for an=an (eng. (inclusive) we are / stay), which consists of the verb 'an' combined with the indefinite pronoun suffix '=an'
        # Both ku=ku (eng. I drink) and e=e (eng. you eat) are not subject to be addressed, as they can be correctly split into the prefix and the verb.
        dic_revise = {"anþ an": "an þan"}

        for key in dic_revise.keys():
            sentence = sentence.replace(key, dic_revise[key]) if key in sentence else sentence

    else: #suffix
        for key in dic_chars.keys():
            sentence = re.sub(re.escape(key) + r'(?=\s|$)', dic_chars[key], sentence)

    sentence = ' '.join(sentence.split())

    return sentence

sentence = 'an=an'

# Clease affixes: process twice for verbs combining both a nominative and an accusative affix (e.g. eci=un=nukar)
for i in range(2):
    sentence = ainAffixCleanse(sentence, dicPrefixes, 0)
    sentence = ainAffixCleanse(sentence, dicSuffixes, 1)

sentence

'an þan'

In [56]:
# Orthography - a=kor itak
dicOrthography = {
    #'ai':'ay',
    #'au':'aw',
    #'ei':'ey',
    #'eu':'ew',
    #'iu':'iw',
    #'oi':'oy',
    #'ou':'ow',
    #'ui':'uy',
    'ch':'c',
    'sh':'s',
}

def char_cleanse(sentence, dic_chars):

    for key in dic_chars.keys():
        sentence = sentence.replace(key, dic_chars[key]) if key in sentence else sentence

    # Remove Extra Spaces
    sentence = ' '.join(sentence.split())
    return sentence

In [57]:
# https://poota.net/archives/872
def normalize_unicode(words):
    unicode_words = ""
    for character in unicodedata.normalize("NFD", words):
        if unicodedata.category(character) != "Mn":
            unicode_words += character
    return unicode_words

In [58]:
def preprocess(sentence):

    # Zenkaku -> Hankaku equivalent
    sentence = normalize_unicode(sentence)

    sentence = sentence.lower()

    # Remove number with parentheses (eg. (123) --> '', [123] --> '')
    sentence = re.sub(r'\(.*?\)|\[.*?\]', '', sentence)

    # Remove symbols
    sentence = re.sub(r"[^a-zþA-Z0-9 =]"," ",sentence)

    # Apply orthography
    sentence = char_cleanse(sentence, dicOrthography)

    # Remove sakehe markers 'v'
    for i in range(5):
        if re.search(r'^v',sentence):
            sentence = re.sub(r"^v","",sentence)

    # Clease affixes: process twice for verbs combining both a nominative and an accusative affix (e.g. eci=un=nukar)
    for i in range(2):
        sentence = ainAffixCleanse(sentence, dicPrefixes, 0)
        sentence = ainAffixCleanse(sentence, dicSuffixes, 1)

    # Trim sentence
    sentence = sentence.strip()

    return sentence

# -- ex. --
sentence = 'vv yayán ainu ku=ne ruwe ne korka, ainu itak ani Transformer[123] ku=kor_ rushuy un!'
preprocess(sentence)

'yayan ainu kuþ ne ruwe ne korka ainu itak ani transformer kuþ kor rusuy un'

In [59]:
texts[1]

'a=ere ukasuy a=e akusu'

In [60]:
# Preprocess sentences
for i in range(len(texts)):
    texts[i] = preprocess(texts[i])

In [61]:
texts[1]

'aþ ere ukasuy aþ e akusu'

In [62]:
# sample
# sentence = 'ク・ネㇷ゚キ　ヒ　カ　エン・ココパンパ　コㇿカ　ネ　ワ　アンペ　カ　ケシㇼキラㇷ゚　ワ　ク・ネㇷ゚キ　ルスイ　ワ　ケシㇼキラㇷ゚　ワ　クス　（ク・ネㇷ゚キ）　チセ　ソイ　ペカ　クネㇷ゚キ　コㇿ　カン。'
# sentence = 'ペウレクﾙ　ネ　コﾛカ　ア・アイヌコﾛ'
sentence = '“Shirokanipe ranran pishkan, konkanipe ranran pishkan.” arian rekpo chiki kane petesoro sapash aine, ainukotan enkashike chikush kor shichorpokun inkarash ko teeta wenkur tane nishpa ne, teeta nishpa tane wenkur ne kotom shiran.'
sentence_orth = preprocess(sentence)

print(sentence)
print(sentence_orth)

“Shirokanipe ranran pishkan, konkanipe ranran pishkan.” arian rekpo chiki kane petesoro sapash aine, ainukotan enkashike chikush kor shichorpokun inkarash ko teeta wenkur tane nishpa ne, teeta nishpa tane wenkur ne kotom shiran.
sirokanipe ranran piskan konkanipe ranran piskan arian rekpo ciki kane petesoro sapas aine ainukotan enkasike cikus kor sicorpokun inkaras ko teeta wenkur tane nispa ne teeta nispa tane wenkur ne kotom siran


#### Affix Marker Check

In [63]:
# 人称接辞マーカー = の両端に文字が残っているものの抽出
for i in range(len(texts)):
    if re.search(r'\S+=\S+', texts[i]):
        print(texts[i])

In [64]:
for i in range(len(texts)):
    if re.search(r'[lqvx]', texts[i]):
        print(texts[i])

#### Save cleansed sentences

In [65]:
# Temporary saving to a txt file
with open("./ain.txt", "w") as output:
    output.write(str(texts))

print(len(texts))

6383


### 2. Prepare Tokenizer

In [66]:
from tokenizers import (
    decoders,
    models,
    normalizers,
    pre_tokenizers,
    processors,
    trainers,
    Tokenizer
)

# Initialize a tokenizer
#tokenizer = ByteLevelBPETokenizer()
tokenizer = Tokenizer(models.WordPiece(unk_token='[UNK]'))

tokenizer.normalizer = normalizers.Sequence(
    [normalizers.Lowercase(), normalizers.NFKC()]
)

tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()

In [67]:
def yield_text():
    for row in texts:
        yield row

In [68]:
trainer = trainers.WordPieceTrainer(
    vocab_size=30_000,
    special_tokens=['[UNK]', '[PAD]', '[CLS]', '[SEP]', '[MASK]'],
    min_frequency=2,
    continuing_subword_prefix='##'
)

In [69]:
tokenizer.train_from_iterator(yield_text(), trainer=trainer)

In [70]:
# Vocabulary size
token_size = tokenizer.get_vocab_size()

In [71]:
cls_id = tokenizer.token_to_id('[CLS]')
sep_id = tokenizer.token_to_id('[SEP]')

In [72]:
tokenizer.post_processor = processors.TemplateProcessing(
    single=f'[CLS]:0 $A:0 [SEP]:0',
    pair=f'[CLS]:0 $A:0 [SEP]:0 $B:1 [SEP]:1',
    special_tokens=[
        ('[CLS]', cls_id),
        ('[SEP]', sep_id)
    ]
)

In [73]:
tokenizer.decoder = decoders.WordPiece(prefix='##')

In [74]:
from transformers import PreTrainedTokenizerFast

full_tokenizer = PreTrainedTokenizerFast(
    tokenizer_object=tokenizer,
    unk_token='[UNK]',
    pad_token='[PAD]',
    cls_token='[CLS]',
    sep_token='[SEP]',
    mask_token='[MASK]'
)

#### Save tokenizer model

In [75]:
full_tokenizer.save_pretrained('ain-base-tokenizer')

('ain-base-tokenizer/tokenizer_config.json',
 'ain-base-tokenizer/special_tokens_map.json',
 'ain-base-tokenizer/tokenizer.json')

#### Check Tokenizer

In [76]:
tokenizer = PreTrainedTokenizerFast.from_pretrained('ain-base-tokenizer')

In [77]:
def tokenizerCheck(sentence: str) -> str:

    # Preprocess
    sentence = preprocess(sentence)
    print('Tokenizer results: ', tokenizer(sentence))

    # Encode sentence
    encodedTokens = tokenizer.encode(sentence)
    print('Encode: ', encodedTokens)

    # Decode token ids to sentences
    decodedTokens = tokenizer.decode(encodedTokens)
    print('Decode: ', decodedTokens)

    # Replace temporary affix markers 'þ' to '='
    decodedTokens_cleansed = re.sub(r'( þ|þ )','=', decodedTokens)
    print('Decode_TokenCleansed: ', decodedTokens_cleansed)

In [78]:
sentence = "ohonno somo unukar=an"

tokenizerCheck(sentence)

Tokenizer results:  {'input_ids': [2, 1313, 141, 473, 100, 56, 3], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1]}
Encode:  [2, 1313, 141, 473, 100, 56, 3]
Decode:  [CLS] ohonno somo unukar þan [SEP]
Decode_TokenCleansed:  [CLS] ohonno somo unukar=an [SEP]


In [79]:
sentence = "Ku=kor wa k=an an pe hinak ta k=anu yakka k=erampewtek"

tokenizerCheck(sentence)

Tokenizer results:  {'input_ids': [2, 233, 58, 55, 342, 54, 54, 67, 781, 68, 342, 746, 148, 342, 1080, 3], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}
Encode:  [2, 233, 58, 55, 342, 54, 54, 67, 781, 68, 342, 746, 148, 342, 1080, 3]
Decode:  [CLS] kuþ kor wa kþ an an pe hinak ta kþ anu yakka kþ erampewtek [SEP]
Decode_TokenCleansed:  [CLS] ku=kor wa k=an an pe hinak ta k=anu yakka k=erampewtek [SEP]


In [80]:
for i in range(token_size):
    #print(tokenizer.decode(i))
    continue